# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access metadata info
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available RecordSets and their fields, using only @id
record_sets = [rs for rs in dataset.metadata.record_sets]
print("Available RecordSets (@id):")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}  (name: {rs.name})")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    - Field @id: {field.id} (name: {field.name}, type: {field.data_type})")

# Show sample records for the first RecordSet
if len(record_sets) > 0:
    first_record_set_id = record_sets[0].id
    print(f"\nSample records from RecordSet {first_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i > 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using their @id
all_record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet {record_set_id}")
    else:
        print(f"No records found for RecordSet {record_set_id}")

# Display DataFrame columns for the first record set with data
non_empty_ids = [rid for rid, df in dataframes.items() if not df.empty]
if non_empty_ids:
    main_record_set_id = non_empty_ids[0]
    print(f"Columns in RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    main_record_set_id = None
    print("No populated record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Example EDA for the main record set
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Select a numeric field by its @id (choose first candidate with numeric values)
    numeric_field_id = None
    for col in df.columns:
        # Try to coerce to float and check for meaningful data
        numeric_vals = pd.to_numeric(df[col], errors='coerce')
        if numeric_vals.notna().sum() > 0 and numeric_vals.dtype in [np.float64, np.int64]:
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        print(f"Using numeric field @id: {numeric_field_id}")

        # Coerce to numeric for calculations
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanmedian(df[numeric_field_id])
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a non-numeric field
        group_field = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                group_field = col
                break
        if group_field:
            print(f"Grouping by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found in DataFrame for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    df[numeric_field_id].hist(bins=30)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id} in RecordSet {main_record_set_id}')
    plt.show()

    if group_field:
        # Boxplot grouped by a categorical field if available
        if df[group_field].nunique() < 20:
            plt.figure(figsize=(12,5))
            df.boxplot(column=numeric_field_id, by=group_field)
            plt.title(f'{numeric_field_id} by {group_field}')
            plt.suptitle("")
            plt.ylabel(numeric_field_id)
            plt.xlabel(group_field)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We examined the available record sets, performed basic EDA including summary statistics and normalizations by referencing fields using their `@id`, and visualized data distributions. For more advanced analyses, refer to the dataset's Croissant schema for deeper data relationships and semantic links.